# J4-02 · Instrumentation fine et RGPD des traces

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 4 · créneau 09h30–10h45 · ⏱️ 1h15 (dont 45 min de pratique)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Instrumenter son propre code avec des observations **typées** (`retriever`, `guardrail`, `evaluator`…) · 2. Signaler une anomalie par le **niveau** d'une observation (`WARNING`, `ERROR`) et son message · 3. Ne pas capturer ce qui ne doit pas l'être (`capture_input=False`) · 4. **Effacer** les traces d'une personne et **purger** les traces anciennes |
| 🧱 **Fil rouge** | Une chaîne « recherche KB → réponse → vérification des sources », entièrement lisible dans Langfuse |
| ✅ **Prérequis** | J4-01 ; KB indexée (J3-02) |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Trois niveaux d'instrumentation

| Niveau | Outil | Ce qu'on obtient |
|---|---|---|
| automatique | `CallbackHandler` dans le `config` LangChain / LangGraph | chaque chaîne, nœud, appel LLM et outil |
| fonction | `@observe(name=…, as_type=…)` | une observation par appel de **votre** fonction, arguments et résultat compris |
| bloc | `with langfuse.start_as_current_observation(name=…, as_type=…) as obs:` | une étape précise, avec ses métadonnées, son niveau, son message |

**Typer** une observation (`retriever`, `guardrail`, `evaluator`, `tool`, `agent`…) change son icône et permet de **filtrer** : « toutes les recherches KB de plus de 2 s », « toutes les vérifications en WARNING ».

## 3. Une chaîne instrumentée de bout en bout

In [ ]:
import re

from langchain_core.messages import HumanMessage, SystemMessage

from helpdesk_ops import kb
from helpdesk_ops.jalons.j1 import decider_par_regles

PROMPT = SystemMessage("Tu es l'agent support N1 de NordTech SA. Réponds en 3 phrases à partir des EXTRAITS seulement, "
                       "en citant leurs références [KB-xxx]. Si les extraits ne répondent pas, dis-le.")


@observe(name="garde-fou", as_type="guardrail")
def garder(question: str) -> str:
    return decider_par_regles(question)


@observe(name="recherche-kb", as_type="retriever", capture_output=False)
def rechercher(question: str):
    docs = kb.rechercher(question, k=3)
    # Sortie tracée : les références et titres, pas le texte complet des articles
    langfuse.update_current_span(output=[{"id": d.metadata["id"], "titre": d.metadata["titre"]} for d in docs])
    return docs


@observe(name="verification-sources", as_type="evaluator")
def verifier(reponse: str, fournies: list[str]) -> dict:
    citees = set(re.findall(r"KB-\d{3}", reponse))
    hors_sources = sorted(citees - set(fournies))
    if not citees or hors_sources:   # anomalie : visible et filtrable dans Langfuse
        langfuse.update_current_span(level="WARNING",
                                     status_message=f"citations absentes ou hors sources : {hors_sources or 'aucune citation'}")
    return {"citees": sorted(citees), "hors_sources": hors_sources}


@observe(name="repondre-instrumente", as_type="agent")
def repondre(question: str) -> str:
    if garder(question) != "SUPPORT":
        return "Je ne peux traiter que les demandes informatiques."
    docs = rechercher(question)
    reponse = llm.invoke([PROMPT, HumanMessage(f"EXTRAITS :\n{kb.formater(docs)}\n\nQUESTION : {question}")],
                         config={"callbacks": [langfuse_handler]}).content
    verifier(reponse, [d.metadata["id"] for d in docs])
    return reponse


with propagate_attributes(session_id="j4-02-instrumentation", tags=["jour-4", "instrumentation"],
                          metadata={"composant": "repondre-instrumente"}, version="instrumentation-v1"):
    for q in ["Ma boîte mail est pleine, je ne reçois plus rien", "Écris-moi un poème"]:
        print(f"❓ {q}\n🤖 {repondre(q)}\n")
langfuse.flush()

**👀 Ce que vous devez observer :** dans Langfuse, la trace `repondre-instrumente` est un arbre **typé** : `garde-fou` (guardrail), `recherche-kb` (retriever, avec les références trouvées), la generation, puis `verification-sources` (evaluator). Si la réponse ne cite rien ou cite un article non fourni, l'évaluateur passe en **WARNING** avec un message : filtrez **Level = WARNING** dans le tableau des traces.

> 💡 **Le « Tips » du Formateur :** l'évaluateur en ligne (`verification-sources`) ne vérifie qu'une **forme** (des références existent et ont été fournies). Il ne dit rien de la **fidélité** du contenu : c'est le travail du juge de J4-04.

## 4. Ne pas capturer ce qui ne doit pas l'être

### [Théorie] Minimiser à la source

Le masque (J1) remplace des motifs connus. Mieux encore : ne pas capturer du tout une entrée qui n'apporte rien au débogage.

In [ ]:
@observe(name="verifier-identite", as_type="tool", capture_input=False)   # les arguments ne partent PAS dans la trace
def verifier_identite(email: str, date_naissance: str) -> bool:
    return email.endswith("@nordtech.example")


with propagate_attributes(session_id="j4-02-instrumentation", tags=["jour-4", "minimisation"]):
    print(verifier_identite("julie.martin@nordtech.example", "1988-04-12"))
langfuse.flush()

**👀 Ce que vous devez observer :** l'observation `verifier-identite` existe (durée, résultat `true`) mais son **input est vide**. La date de naissance n'a jamais quitté le notebook.

| Mesure | Quand l'utiliser |
|---|---|
| `capture_input=False` / `capture_output=False` | l'argument ou le résultat n'aide pas au débogage et contient des données sensibles |
| `mask=` du client (J1) | du texte libre (tickets, messages) qui peut contenir des e-mails, téléphones, IBAN |
| sortie résumée (comme `rechercher` ci-dessus) | un gros objet : on trace les identifiants, pas le contenu |

## 5. 🔒 RGPD : effacer et purger les traces

### [Théorie] Le droit à l'effacement s'applique aussi à Langfuse

En J1 et J2, une question était restée ouverte : la *data retention* automatique de Langfuse est réservée à l'édition Enterprise ; peut-on effacer en open source ? **Oui, par l'API** : vérifié en préparant ce jour (une trace supprimée, ses observations disparaissent). `helpdesk_ops.observabilite` fournit :

| Fonction | Usage |
|---|---|
| `traces_concernees(langfuse, avant=…, user_id=…, session_id=…)` | lister les traces visées |
| `purger_traces(…, simulation=True)` | lister sans supprimer (à journaliser) ; `simulation=False` : supprimer |

Un critère est **obligatoire** : la fonction refuse de « tout purger » par mégarde.

In [ ]:
import time
from datetime import datetime, timedelta, timezone

import httpx

from helpdesk_ops.observabilite import purger_traces, traces_concernees

# Une conversation de démonstration, avec des données personnelles, que la personne demande ensuite d'effacer
SESSION = "j4-02-demande-effacement"
with propagate_attributes(session_id=SESSION, user_id="collab-demo-rgpd", tags=["jour-4", "rgpd"]):
    with langfuse.start_as_current_observation(name="demande", as_type="span",
                                               input="Bonjour, Julie Martin, mon numéro est le 0470 12 34 56") as obs:
        obs.update(output="ticket NT-0099 créé")
langfuse.flush()
time.sleep(10)   # le temps que le serveur ingère la trace

print("Traces de la session :", traces_concernees(langfuse, session_id=SESSION))
print("Simulation          :", purger_traces(langfuse, session_id=SESSION))

In [ ]:
supprimees = purger_traces(langfuse, session_id=SESSION, simulation=False)
print("Supprimées :", supprimees)
time.sleep(15)
print("Traces restantes pour la session :", traces_concernees(langfuse, session_id=SESSION))

# Purge planifiée : ce qu'une conservation de 30 jours supprimerait aujourd'hui (simulation)
il_y_a_30_jours = datetime.now(timezone.utc) - timedelta(days=30)
print("Purge à 30 jours (simulation) :", len(purger_traces(langfuse, avant=il_y_a_30_jours)), "trace(s)")

**👀 Ce que vous devez observer :** la trace de la session de démonstration est listée, supprimée, puis introuvable ; la purge à 30 jours ne trouve rien (les traces du cours ont moins de 30 jours).

**L'effacement complet de Julie**, c'est donc : les checkpoints (`delete_thread`, J2), les **traces** (`purger_traces(user_id=…)` ou `session_id=…`), le ticketing, les sauvegardes et les journaux. Une procédure écrite, testée, et une durée de conservation inscrite au registre des traitements.

> 🔒 **Qui voit les traces ?** Tout utilisateur de l'interface Langfuse voit les conversations tracées. Limitez les comptes au strict nécessaire, et vérifiez les rôles disponibles dans votre édition. Le masquage et la minimisation réduisent ce qu'un accès indu exposerait.

## 🧭 Récap

| Vous savez maintenant… | API |
|---|---|
| Typer vos observations | `@observe(name=…, as_type="retriever" \| "guardrail" \| "evaluator" \| …)` |
| Signaler une anomalie | `langfuse.update_current_span(level="WARNING", status_message=…)` |
| Minimiser | `@observe(capture_input=False)`, sortie résumée, `mask=` |
| Effacer / purger | `purger_traces(langfuse, user_id=… \| session_id=… \| avant=…, simulation=False)` |

### ✅ Quiz express

1. Quel intérêt à typer une observation `retriever` plutôt que `span` ?
2. Une vérification passe en WARNING : qu'est-ce que cela change pour l'utilisateur ? Pour l'équipe ?
3. Quand préférer `capture_input=False` au masquage ?
4. La *data retention* étant une fonction Enterprise, comment respecter une durée de conservation en open source ?

<details>
<summary>Réponses</summary>

1. Icône, filtres et indicateurs dédiés : on retrouve toutes les recherches, leur latence, leurs résultats.
2. Rien pour l'utilisateur (la réponse est partie) ; pour l'équipe, une anomalie filtrable et mesurable, à transformer en score ou en alerte.
3. Quand l'argument n'aide pas au débogage : mieux vaut ne rien envoyer que masquer imparfaitement.
4. Par une purge planifiée via l'API (`purger_traces(avant=…)`), journalisée, avec une durée inscrite au registre.
</details>